In [9]:
# =========================
# 1. IMPORT THƯ VIỆN
# =========================
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [10]:
# =========================
# 2. KHAI BÁO FILE & CỘT
# =========================
INPUT_PATH = Path(r"D:\User\Tài liệu học\HK2 - 2025-2026\1. Data\4. Translator\processed_data_translated_en.csv")

REVIEW_ID_COL = "review_id"
TEXT_COL = "Comment_en"

OUTPUT_PATH = INPUT_PATH.with_name(INPUT_PATH.stem + "_split_clauses.csv")

In [11]:
# =========================
# 3. CẤU HÌNH TÁCH CLAUSE
# =========================
MIN_WORDS_AFTER_SPLIT = 4
MIN_WORDS_STANDALONE = 4

LIST_BREAK = "<LIST_BREAK>"
DOT = "<DOT>"

ABBREVIATIONS = [
    "e.g.", "i.e.", "etc.", "mr.", "mrs.", "ms.", "dr.", "prof.", "sr.", "jr.",
    "vs.", "st.", "no.", "approx.", "min.", "max.", "a.m.", "p.m.", "u.s.", "u.k."
]

CONNECTORS = [
    # Nhóm chuyển ý / đối lập mạnh, nên tách
    "on the other hand",
    "in contrast",
    "however",
    "nevertheless",
    "nonetheless",
    "instead",
    "otherwise",

    # Nhóm đối lập nhẹ, có thể tách nếu hai vế đủ nghĩa
    "although",
    "though",
    "even though",
    "even if",
    "whereas",
    "while",

    # Nhóm nối hai ý khác hướng rõ
    "but",
    "yet",

    # Nhóm chuyển bối cảnh thời gian / diễn biến
    "meanwhile",
]

CONNECTORS_SORTED = sorted(CONNECTORS, key=len, reverse=True)

CONNECTOR_RE = re.compile(
    r"(?:[,;:]\s*|\s+)\b("
    + "|".join(re.escape(c) for c in CONNECTORS_SORTED)
    + r")\b,?\s+",
    flags=re.IGNORECASE
)

LEADING_CONNECTOR_RE = re.compile(
    r"^\s*("
    + "|".join(re.escape(c) for c in CONNECTORS_SORTED)
    + r")\b,?\s*",
    flags=re.IGNORECASE
)

WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+")

AUX_AND_COMMON_VERBS = {
    "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did",
    "can", "could", "will", "would", "shall", "should", "may", "might", "must",

    "go", "goes", "went", "gone",
    "come", "comes", "came",
    "visit", "visits", "visited",
    "see", "sees", "saw", "seen",
    "enjoy", "enjoys", "enjoyed",
    "like", "likes", "liked",
    "love", "loves", "loved",
    "recommend", "recommends", "recommended",
    "return", "returns", "returned",
    "buy", "buys", "bought",
    "pay", "pays", "paid",
    "cost", "costs",
    "feel", "feels", "felt",
    "make", "makes", "made",
    "get", "gets", "got",
    "take", "takes", "took", "taken",
    "spend", "spends", "spent",
    "wait", "waits", "waited",
    "walk", "walks", "walked",
    "drive", "drives", "drove",
    "rent", "rents", "rented",
    "book", "books", "booked",
    "try", "tries", "tried",
    "eat", "eats", "ate", "eaten",
    "drink", "drinks", "drank",
    "stay", "stays", "stayed",
    "look", "looks", "looked",
    "seem", "seems", "seemed",
    "become", "becomes", "became",
    "bring", "brings", "brought",
    "know", "knows", "knew",
    "think", "thinks", "thought",
}

NON_VERB_ING_WORDS = {
    "amazing", "interesting", "boring", "stunning", "charming", "fascinating",
    "relaxing", "disappointing", "confusing", "annoying", "overwhelming",
    "outstanding", "surrounding"
}


In [12]:
# =========================
# 4. CÁC HÀM XỬ LÝ
# =========================
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\r", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_clause_text(text):
    text = "" if text is None else str(text)
    text = text.replace(DOT, ".")
    text = re.sub(r"\s+", " ", text).strip()

    # Xóa marker liệt kê ở đầu clause: 1. / 1) / 1,
    text = re.sub(r"^\s*\d{1,2}[\.\),]\s*", "", text)

    text = text.strip(" \t-–—,;:")
    return text


def protect_periods(text):
    # Bảo vệ số thập phân: 4.5
    text = re.sub(r"(\d)\.(\d)", rf"\1{DOT}\2", text)

    # Bảo vệ viết tắt: e.g., i.e., etc.
    for abbr in ABBREVIATIONS:
        pattern = re.compile(re.escape(abbr), flags=re.IGNORECASE)
        text = pattern.sub(lambda m: m.group(0).replace(".", DOT), text)

    return text


def mark_list_items(text):
    """
    Đánh dấu dạng liệt kê:
    1. text
    1) text
    1, text
    """
    text = re.sub(
        r"(^|[\s;|])(\d{1,2})[\.\)]\s+",
        lambda m: f"{m.group(1)} {LIST_BREAK} {m.group(2)}) ",
        text
    )

    text = re.sub(
        r"(^|[\s;|])(\d{1,2}),\s+(?=[A-Za-z])",
        lambda m: f"{m.group(1)} {LIST_BREAK} {m.group(2)}) ",
        text
    )

    return text.strip()


def split_into_sentences(text):
    protected = protect_periods(text)

    pieces = re.split(
        r"(?<=[.!?])\s+(?=(?:[\"'(\[])?[A-Za-z0-9])",
        protected
    )

    return [clean_clause_text(p) for p in pieces if clean_clause_text(p)]


def get_words(text):
    return WORD_RE.findall(text or "")


def has_verb_like(text):
    words = [w.lower() for w in get_words(text)]

    if any(w in AUX_AND_COMMON_VERBS for w in words):
        return True

    for w in words:
        if len(w) > 4 and w.endswith("ed"):
            return True

        if len(w) > 5 and w.endswith("ing") and w not in NON_VERB_ING_WORDS:
            return True

    return False


def is_reasonable_clause(text, min_words=4):
    text = clean_clause_text(text)
    words = get_words(text)

    if len(words) < min_words:
        return False

    if has_verb_like(text):
        return True

    # Cho phép cụm review rút gọn nhưng vẫn đủ nghĩa
    # Ví dụ: beautiful beach with clear water
    if len(words) >= 7:
        return True

    return False


def strip_leading_connector(text):
    return clean_clause_text(LEADING_CONNECTOR_RE.sub("", text))


def is_bad_connector_use(text, match, connector):
    right_part = text[match.start():match.end() + 30].lower()

    # Tránh tách "not only ... but also"
    if connector.lower() == "but" and re.match(r"^\s*,?\s*but\s+also\b", right_part):
        return True

    # Tránh tách "so beautiful", "so good", "so many"...
    if connector.lower() == "so" and re.match(
        r"^\s*,?\s*so\s+(good|nice|beautiful|amazing|bad|hot|cold|much|many|far|long|short)\b",
        right_part
    ):
        return True

    return False


def recursive_connector_split(text, depth=0, max_depth=10):
    text = clean_clause_text(text)

    if not text:
        return []

    if depth >= max_depth:
        return [text]

    for m in CONNECTOR_RE.finditer(text):
        connector = m.group(1).lower()

        if is_bad_connector_use(text, m, connector):
            continue

        left = clean_clause_text(text[:m.start()])
        right = clean_clause_text(text[m.start():])
        right_core = strip_leading_connector(right)

        # Chỉ tách nếu cả 2 vế đều không bị cụt
        if (
            is_reasonable_clause(left, min_words=MIN_WORDS_AFTER_SPLIT)
            and is_reasonable_clause(right_core, min_words=MIN_WORDS_AFTER_SPLIT)
        ):
            return (
                recursive_connector_split(left, depth=depth + 1, max_depth=max_depth)
                + recursive_connector_split(right, depth=depth + 1, max_depth=max_depth)
            )

    return [text]


def join_two_clauses(a, b):
    a = clean_clause_text(a)
    b = clean_clause_text(b)

    if not a:
        return b
    if not b:
        return a

    if a.endswith((".", "!", "?")):
        return f"{a} {b}"

    return f"{a}, {b}"


def merge_short_fragments(candidates):
    candidates = [clean_clause_text(c) for c in candidates if clean_clause_text(c)]

    if len(candidates) <= 1:
        return candidates

    merged = []

    for c in candidates:
        if not merged:
            merged.append(c)
            continue

        if not is_reasonable_clause(c, min_words=MIN_WORDS_STANDALONE):
            merged[-1] = join_two_clauses(merged[-1], c)
        else:
            merged.append(c)

    # Nếu clause đầu tiên vẫn quá cụt thì gộp vào clause sau
    while len(merged) > 1 and not is_reasonable_clause(merged[0], min_words=MIN_WORDS_STANDALONE):
        merged[1] = join_two_clauses(merged[0], merged[1])
        merged.pop(0)

    return merged


def split_text_to_clauses(text):
    text = normalize_text(text)

    if not text:
        return []

    text = mark_list_items(text)

    list_parts = [
        p for p in text.split(LIST_BREAK)
        if clean_clause_text(p)
    ]

    all_candidates = []

    for part in list_parts:
        part = clean_clause_text(part)

        for sent in split_into_sentences(part):
            clause_candidates = recursive_connector_split(sent)
            clause_candidates = merge_short_fragments(clause_candidates)
            all_candidates.extend(clause_candidates)

    all_candidates = merge_short_fragments(all_candidates)

    return all_candidates if all_candidates else [text]


def idx_to_letters(idx):
    """
    0 -> a
    1 -> b
    25 -> z
    26 -> aa
    """
    idx += 1
    letters = ""

    while idx > 0:
        idx, rem = divmod(idx - 1, 26)
        letters = chr(97 + rem) + letters

    return letters


def clean_review_id(x):
    if pd.isna(x):
        return ""

    if isinstance(x, float) and x.is_integer():
        return str(int(x))

    return str(x).strip()

In [13]:
# =========================
# 5. ĐỌC FILE
# =========================
df = pd.read_csv(INPUT_PATH)

print("Các cột trong file:")
print(df.columns.tolist())

if REVIEW_ID_COL not in df.columns:
    raise ValueError(f"Không tìm thấy cột '{REVIEW_ID_COL}' trong file.")

if TEXT_COL not in df.columns:
    raise ValueError(f"Không tìm thấy cột '{TEXT_COL}' trong file.")

Các cột trong file:
['review_id', 'Username', 'Address', 'Rating', 'Language', 'Comment', 'Comment_en', 'translation_status', 'translation_model']


In [14]:
# =========================
# 6. TÁCH CLAUSE
# =========================
rows = []
clause_counter_by_review = defaultdict(int)

for _, row in df.iterrows():
    review_id = clean_review_id(row[REVIEW_ID_COL])
    original_text = row[TEXT_COL]

    clauses = split_text_to_clauses(original_text)

    for clause in clauses:
        clause_counter_by_review[review_id] += 1
        clause_order = clause_counter_by_review[review_id]

        new_row = row.to_dict()

        # ID dạng 1.a, 1.b, 1.c
        new_row["ID_clause"] = f"{review_id}.{idx_to_letters(clause_order - 1)}"

        # Clause sau khi tách
        new_row["Clause"] = clause

        rows.append(new_row)

df_out = pd.DataFrame(rows)

In [15]:
# =========================
# 7. SẮP XẾP LẠI THỨ TỰ CỘT
# =========================
keep_cols = [
    "review_id",
    "Username",
    "Address",
    "Rating",
    "Language",
    "ID_clause",
    "Clause",
]

keep_cols = [c for c in keep_cols if c in df_out.columns]

df_out = df_out[keep_cols]

In [16]:
# =========================
# 8. LƯU FILE
# =========================
df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Đã tách clause xong.")
print("Số dòng ban đầu:", len(df))
print("Số dòng sau khi tách clause:", len(df_out))
print("File đã lưu tại:", OUTPUT_PATH.resolve())

display(df_out[["review_id", "ID_clause", "Clause"]].head(30))

Đã tách clause xong.
Số dòng ban đầu: 64406
Số dòng sau khi tách clause: 231726
File đã lưu tại: D:\User\Tài liệu học\HK2 - 2025-2026\1. Data\4. Translator\processed_data_translated_en_split_clauses.csv


,review_id,ID_clause,Clause
0,1,1.a,It's about 10-15 minutes away from the old cit...
1,1,1.b,"The beach is wide, with soft sand and a long c..."
2,1,1.c,It's quite calm and there's no feeling of a cr...
3,1,1.d,There's a lot of cozy cafes and restaurants al...
4,1,1.e,"By the way, many cafes have a bed and an umbre..."
5,2,2.a,Quiet and quaint. Very laid back vive
6,3,3.a,"It's a wonderful calm beach with clear water, ..."
7,3,3.b,but you're not seeing sunset.
8,4,4.a,"While spending time in Hoi An, I was looking f..."
9,4,4.b,"As an expat, it’s not always easy to find Scan..."
